# ⚡ Azure OpenAI(Responses API)와 함께하는 동시 에이전트 워크플로우 (.NET)

## 📋 고성능 병렬 처리 튜토리얼

이 노트북은 Microsoft Agent Framework for .NET과 Azure OpenAI(Responses API)를 사용한 <strong>동시 워크플로우 패턴</strong>을 보여줍니다. 여러 AI 에이전트를 동시에 실행하면서 조정과 데이터 일관성을 유지해 처리량을 극대화하는 고성능 병렬 처리 워크플로우를 구축하는 방법을 배웁니다.

> **마이그레이션 노트:** 이 샘플은 이전에 GitHub 모델을 사용했습니다. GitHub 모델은 사용 중단 예정(2026년 7월 종료)이며 Responses API를 지원하지 않으므로, 현재는 `AzureOpenAIClient.GetOpenAIResponseClient(...)`를 통해 <strong>Responses API</strong>가 포함된 <strong>Azure OpenAI</strong>를 사용합니다.

## 🎯 학습 목표

### 🚀 **동시 처리 기본**
- **병렬 에이전트 실행**: 최대 성능을 위해 여러 AI 에이전트를 동시에 실행
- **Async/Await 패턴**: 효율적인 동시성을 위한 .NET의 비동기 프로그래밍 모델 활용
- **Azure OpenAI Responses API 통합**: Azure OpenAI Responses API에 대한 여러 동시 호출 조정
- **리소스 관리**: 동시 작업 간에 AI 모델 리소스를 효율적으로 관리

### 🏗️ **고급 동시성 아키텍처**
- **작업 기반 병렬성**: 최적의 동시 실행을 위한 .NET Task Parallel Library 활용
- **동기화 패턴**: 경쟁 조건을 피하면서 동시 에이전트 조정
- **부하 분산**: 사용 가능한 동시 처리 용량에 작업을 효율적으로 분배
- <strong>내결함성</strong>: 전체 워크플로우를 중단 없이 개별 에이전트 실패 처리

### 🏢 **엔터프라이즈 동시성 애플리케이션**
- **대용량 문서 처리**: 여러 문서를 동시에 처리
- **실시간 콘텐츠 분석**: 들어오는 데이터 스트림의 동시 분석
- **배치 처리 최적화**: 대규모 데이터 처리 작업 처리량 극대화
- **다중 모드 분석**: 서로 다른 콘텐츠 유형과 포맷 병렬 처리

## ⚙️ 사전 준비 및 설정

### 📦 **필수 NuGet 패키지**

고성능 동시 워크플로우를 위한 필수 패키지:

```xml
<!-- Core AI Framework with Async Support -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.1" />

<!-- Azure OpenAI client (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />

<!-- Azure Identity and Async LINQ for Advanced Operations -->
<PackageReference Include="Azure.Identity" Version="1.15.0" />
<PackageReference Include="System.Linq.Async" Version="6.0.3" />

<!-- Agent Framework -->
<!-- Microsoft.Agents.AI - Core agent abstractions with async support -->
<!-- Microsoft.Agents.AI.OpenAI - Azure OpenAI Responses integration with concurrency -->
```

### 🔑 **Azure OpenAI 구성**

`AzureCliCredential`이 인증할 수 있도록 Azure CLI(`az login`)로 로그인한 후 Azure OpenAI 리소스 세부정보를 설정하세요. Responses API는 안정적인 `/openai/v1/` 엔드포인트를 사용하므로 `api-version` 관리는 필요 없습니다.

**환경 설정 (.env 파일):**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**동시 처리 고려사항:**
```csharp
// Configure connection pooling / timeouts for concurrent requests
var clientOptions = new AzureOpenAIClientOptions()
{
    NetworkTimeout = TimeSpan.FromMinutes(5)
};
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential(), clientOptions);
```

### 🏗️ **동시 워크플로우 아키텍처**

```mermaid
graph TD
    A[워크플로우 입력] --> B[작업 분배]
    B --> C[동시 에이전트 풀]
    C --> D[에이전트 작업 1]
    C --> E[에이전트 작업 2]
    C --> F[에이전트 작업 3]
    C --> G[에이전트 작업 N]
    
    D --> H[결과 집계]
    E --> H
    F --> H
    G --> H
    
    H --> I[동기화된 출력]
    
    J[Azure OpenAI 응답 API] --> D
    J --> E
    J --> F
    J --> G
    
    K[".NET 작업 스케줄러"] --> C
```

**핵심 구성요소:**
- **작업 병렬 라이브러리**: 동시 작업을 위한 .NET 내장 지원
- **에이전트 풀**: 병렬 처리를 위한 다중 에이전트 인스턴스
- **결과 집계**: 동시 에이전트 결과의 조정 및 병합
- **동기화 지점**: 동시 작업 전체에서 데이터 일관성 보장

## 🎨 **동시 워크플로우 디자인 패턴**

### 🔍 **병렬 연구 및 분석**
```
Research Topic → Concurrent Research Agents → Result Synthesis → Final Report
```

### 📊 **다중 소스 데이터 처리**
```
Data Sources → Parallel Processing Agents → Data Integration → Unified Output
```

### 🎭 **콘텐츠 생성 파이프라인**
```
Content Requirements → Concurrent Content Generators → Quality Review → Final Content
```

### 🔄 **팬아웃/팬인 처리**
```
Single Input → Multiple Concurrent Processors → Result Aggregation → Single Output
```

## 🏢 **엔터프라이즈 성능 이점**

### ⚡ **처리량 및 확장성**
- **선형 성능 확장**: 처리량 증가를 위해 더 많은 동시 에이전트 추가
- **리소스 활용도**: 사용 가능한 AI 모델 용량의 최대 효율성
- **처리 시간 감소**: 병렬 실행을 통한 상당한 시간 절감
- **탄력적 확장**: 작업 부하에 따라 동시 에이전트 수 동적 조정

### 🛡️ **신뢰성 및 복원력**
- **고장 격리**: 개별 에이전트 실패가 다른 동시 작업에 영향 없음
- **우아한 저하**: 에이전트 용량이 줄어들어도 시스템 계속 운영
- **오류 복구**: 실패한 동시 작업을 위한 자동 재시도 메커니즘
- **부하 분산**: 이용 가능한 에이전트 간 작업 고른 분배

### 📊 **성능 모니터링**
- **동시 실행 지표**: 모든 병렬 작업의 성능 추적
- **리소스 사용 분석**: CPU, 메모리, 네트워크 활용도 모니터링
- **처리량 분석**: 동시 처리로 인한 효율성 향상 측정
- **병목 현상 감지**: 성능 제약 사항 식별 및 해결

### 🔧 **개발 및 운영**
- **비동기 프로그래밍 모델**: .NET의 성숙한 async/await 패턴 활용
- **작업 조정**: 내장된 작업 관리 및 조정 기능
- **예외 처리**: 동시 작업에 대한 포괄적 오류 처리
- **디버깅 지원**: 동시 워크플로우를 위한 Visual Studio 디버깅 도구

.NET으로 고성능 동시 AI 워크플로우를 구축해 봅시다! 🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [4]:
#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:
#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [6]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [7]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using Microsoft.Agents.AI.Workflows.Reflection;

In [9]:
 using DotNetEnv;

In [10]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [14]:
const string ResearcherAgentName = "Researcher-Agent";
const string ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction.";

In [15]:
const string PlanAgentName = "Plan-Agent";
const string PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings.";

In [ ]:
AIAgent researcherAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ResearcherAgentName,instructions:ResearcherAgentInstructions);
AIAgent plannerAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:PlanAgentName,instructions:PlanAgentInstructions);

In [17]:

public class ConcurrentStartExecutor() :
    ReflectingExecutor<ConcurrentStartExecutor>("ConcurrentStartExecutor"),
    IMessageHandler<string>
{
    /// <summary>
    /// Starts the concurrent processing by sending messages to the agents.
    /// </summary>
    /// <param name="message">The user message to process</param>
    /// <param name="context">Workflow context for accessing workflow services and adding events</param>
    /// <returns>A task representing the asynchronous operation</returns>
    public async ValueTask HandleAsync(string message, IWorkflowContext context)
    {
        // Broadcast the message to all connected agents. Receiving agents will queue
        // the message but will not start processing until they receive a turn token.
        await context.SendMessageAsync(new ChatMessage(ChatRole.User, message));
        // Broadcast the turn token to kick off the agents.
        await context.SendMessageAsync(new TurnToken(emitEvents: true));
    }
}

/// <summary>
/// Executor that aggregates the results from the concurrent agents.
/// </summary>
public class ConcurrentAggregationExecutor() :
    ReflectingExecutor<ConcurrentAggregationExecutor>("ConcurrentAggregationExecutor"),
    IMessageHandler<ChatMessage>
{
    private readonly List<ChatMessage> _messages = [];

    /// <summary>
    /// Handles incoming messages from the agents and aggregates their responses.
    /// </summary>
    /// <param name="message">The message from the agent</param>
    /// <param name="context">Workflow context for accessing workflow services and adding events</param>
    /// <returns>A task representing the asynchronous operation</returns>
    public async ValueTask HandleAsync(ChatMessage message, IWorkflowContext context)
    {
        this._messages.Add(message);

        if (this._messages.Count == 2)
        {
            var formattedMessages = string.Join(Environment.NewLine, this._messages.Select(m => $"{m.AuthorName}: {m.Text}"));
            await context.YieldOutputAsync(formattedMessages);
        }
    }
}

In [18]:
var startExecutor = new ConcurrentStartExecutor();
var aggregationExecutor = new ConcurrentAggregationExecutor();

In [19]:
var workflow = new WorkflowBuilder(startExecutor)
            .AddFanOutEdge(startExecutor, targets: [researcherAgent, plannerAgent])
            .AddFanInEdge(aggregationExecutor, sources: [researcherAgent, plannerAgent])
            .WithOutputFrom(aggregationExecutor)
            .Build();

In [20]:

        StreamingRun run = await InProcessExecution.StreamAsync(workflow, "Plan a trip to Seattle in December");
        await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
        {
            if (evt is WorkflowOutputEvent output)
            {
                Console.WriteLine($"Workflow completed with results:\n{output.Data}");
            }
        }

Workflow completed with results:
Plan-Agent: December is a magical time to visit Seattle, as the city embraces the festive season with sparkling holiday lights, seasonal activities, cozy indoor attractions, and hearty cuisine. The weather will be chilly, often rainy, and occasionally snowy, so pack accordingly. Here's a detailed trip plan for your Seattle visit:

---

### **Travel Dates**  
Suggested schedule: **3-5 days in Seattle (example: December 15–19)**  
Adjust according to your preferences and availability.

---

### **Packing Essentials**  
- Warm, waterproof coat  
- Umbrella or rain jacket (Seattle has rainy winters)  
- Waterproof boots or shoes  
- Layers: sweaters, thermal tops, scarves, gloves, and hats  
- Day backpack for exploring  
- Travel charger and portable power bank  
- Camera or phone for holiday photos  

---

### **Day 1: Arrival and Exploring Downtown**  
**Morning**  
- Arrive at **Seattle-Tacoma International Airport (SEA)**.  
- Transfer to your accommod

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
